# Pose Tracker — MediaPipe (API Tasks)
### Fisioterapia · Ergonomía · Deporte

Captura **postura, ángulos articulares y velocidad de movimiento** desde un **video grabado** o desde la **webcam**, y exporta un CSV para el dashboard.

> Usa la API nueva `PoseLandmarker` (MediaPipe Tasks). La API antigua `mp.solutions` ya no existe en las versiones actuales.
> Los 33 landmarks son los mismos, así que los ángulos y rangos no cambian.

**Ejecuta en orden.** Después de la celda 1, reinicia el kernel si es la primera vez que instalas.

In [ ]:
%pip install -q -U mediapipe opencv-python pandas matplotlib

## 2. Imports y configuración

In [ ]:
import os, time, urllib.request
from datetime import datetime, timedelta
from collections import deque

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mediapipe as mp

V = mp.tasks.vision
BaseOptions = mp.tasks.BaseOptions

# ---------------- CONFIGURACIÓN ----------------
ENFOQUE = "fisioterapia"      # "fisioterapia" | "ergonomia" | "deporte"
VIDEO_PATH = "sesion.mp4"     # ruta a un video; pon None para usar la webcam (solo en local)
CAMERA_INDEX = 0
MODEL_VARIANT = "full"        # "lite" (rápido) | "full" (equilibrado) | "heavy" (más preciso)
SHOW_PREVIEW = False          # True solo si corres en local con ventana gráfica
GUARDAR_VIDEO_ANOTADO = True  # guarda un mp4 con el esqueleto y los ángulos
VELOCITY_WINDOW = 5           # frames para suavizar la velocidad
CONFIANZA = 0.6

STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
CSV_FILENAME = f"pose_log_{STAMP}.csv"
VIDEO_SALIDA = f"pose_anotado_{STAMP}.mp4"
print("Enfoque:", ENFOQUE, "| Fuente:", VIDEO_PATH or f"webcam {CAMERA_INDEX}")

## 3. Descarga del modelo

La API nueva necesita un archivo `.task`. Se descarga una sola vez. Si tu entorno no tiene internet, súbelo manualmente con ese nombre.

In [ ]:
MODEL_URL = ("https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
             f"pose_landmarker_{MODEL_VARIANT}/float16/latest/pose_landmarker_{MODEL_VARIANT}.task")
MODEL_PATH = f"pose_landmarker_{MODEL_VARIANT}.task"

if not os.path.exists(MODEL_PATH):
    print("Descargando modelo...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("Modelo listo:", MODEL_PATH, f"({os.path.getsize(MODEL_PATH)/1e6:.1f} MB)")

## 4. Articulaciones a medir

In [ ]:
LANDMARKS = V.PoseLandmark

# (punto_A, vértice, punto_B): el ángulo se mide en el vértice
JOINT_ANGLES = {
    "codo_izq":    (LANDMARKS.LEFT_SHOULDER,  LANDMARKS.LEFT_ELBOW,   LANDMARKS.LEFT_WRIST),
    "codo_der":    (LANDMARKS.RIGHT_SHOULDER, LANDMARKS.RIGHT_ELBOW,  LANDMARKS.RIGHT_WRIST),
    "hombro_izq":  (LANDMARKS.LEFT_ELBOW,     LANDMARKS.LEFT_SHOULDER, LANDMARKS.LEFT_HIP),
    "hombro_der":  (LANDMARKS.RIGHT_ELBOW,    LANDMARKS.RIGHT_SHOULDER, LANDMARKS.RIGHT_HIP),
    "cadera_izq":  (LANDMARKS.LEFT_SHOULDER,  LANDMARKS.LEFT_HIP,     LANDMARKS.LEFT_KNEE),
    "cadera_der":  (LANDMARKS.RIGHT_SHOULDER, LANDMARKS.RIGHT_HIP,    LANDMARKS.RIGHT_KNEE),
    "rodilla_izq": (LANDMARKS.LEFT_HIP,       LANDMARKS.LEFT_KNEE,    LANDMARKS.LEFT_ANKLE),
    "rodilla_der": (LANDMARKS.RIGHT_HIP,      LANDMARKS.RIGHT_KNEE,   LANDMARKS.RIGHT_ANKLE),
    "cuello":      (LANDMARKS.LEFT_SHOULDER,  LANDMARKS.NOSE,         LANDMARKS.RIGHT_SHOULDER),
    "tobillo_izq": (LANDMARKS.LEFT_KNEE,      LANDMARKS.LEFT_ANKLE,   LANDMARKS.LEFT_FOOT_INDEX),
    "tobillo_der": (LANDMARKS.RIGHT_KNEE,     LANDMARKS.RIGHT_ANKLE,  LANDMARKS.RIGHT_FOOT_INDEX),
}

## 5. Rangos de referencia por enfoque

- **Fisioterapia**: rango de movimiento (ROM) funcional
- **Ergonomía**: ángulos posturales seguros en postura sostenida
- **Deporte**: rangos técnicos amplios

> Son valores de partida **sin validación clínica**. Deben revisarlos un fisioterapeuta y/o un ergónomo antes de uso con pacientes.

In [ ]:
RANGOS_REFERENCIA = {
    "fisioterapia": {
        "codo_izq": (30, 150), "codo_der": (30, 150),
        "hombro_izq": (0, 170), "hombro_der": (0, 170),
        "cadera_izq": (80, 170), "cadera_der": (80, 170),
        "rodilla_izq": (60, 175), "rodilla_der": (60, 175),
        "cuello": (150, 180),
        "tobillo_izq": (70, 130), "tobillo_der": (70, 130),
    },
    "ergonomia": {
        "codo_izq": (80, 120), "codo_der": (80, 120),
        "hombro_izq": (0, 45), "hombro_der": (0, 45),
        "cadera_izq": (90, 135), "cadera_der": (90, 135),
        "rodilla_izq": (90, 135), "rodilla_der": (90, 135),
        "cuello": (160, 180),
        "tobillo_izq": (80, 110), "tobillo_der": (80, 110),
    },
    "deporte": {
        "codo_izq": (10, 175), "codo_der": (10, 175),
        "hombro_izq": (0, 180), "hombro_der": (0, 180),
        "cadera_izq": (40, 180), "cadera_der": (40, 180),
        "rodilla_izq": (30, 180), "rodilla_der": (30, 180),
        "cuello": (140, 180),
        "tobillo_izq": (60, 140), "tobillo_der": (60, 140),
    },
}

def validar_angulo(nombre, valor, enfoque):
    """'ok', 'alerta' (±15% del rango) o 'fuera_rango'."""
    rangos = RANGOS_REFERENCIA.get(enfoque, {})
    if nombre not in rangos or valor is None:
        return "sin_dato"
    lo, hi = rangos[nombre]
    if lo <= valor <= hi:
        return "ok"
    margen = (hi - lo) * 0.15
    if (lo - margen) <= valor <= (hi + margen):
        return "alerta"
    return "fuera_rango"

## 6. Funciones de cálculo

In [ ]:
def calcular_angulo(a, b, c):
    """Ángulo en grados en el punto b, formado por a-b-c."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc = a - b, c - b
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos, -1.0, 1.0))))

def get_coords(lms, idx, w, h):
    lm = lms[idx.value]
    return (lm.x * w, lm.y * h)

def calcular_velocidad(hist):
    """Velocidad (px/s) = recorrido en la ventana / tiempo transcurrido. hist = [(t, x, y), ...]"""
    if len(hist) < 2:
        return 0.0
    ts = [p[0] for p in hist]
    pts = np.array([[p[1], p[2]] for p in hist])
    dt = ts[-1] - ts[0]
    if dt <= 0:
        return 0.0
    recorrido = np.linalg.norm(np.diff(pts, axis=0), axis=1).sum()
    return float(recorrido / dt)

## 7. Procesamiento (video o webcam)

In [ ]:
COLORES = {"ok": (0, 200, 0), "alerta": (0, 200, 255), "fuera_rango": (0, 0, 255), "sin_dato": (180, 180, 180)}

def crear_landmarker():
    opts = V.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=V.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=CONFIANZA,
        min_pose_presence_confidence=CONFIANZA,
        min_tracking_confidence=CONFIANZA,
    )
    return V.PoseLandmarker.create_from_options(opts)


def procesar(fuente=VIDEO_PATH, enfoque=ENFOQUE, mostrar=SHOW_PREVIEW, guardar_video=GUARDAR_VIDEO_ANOTADO):
    global CSV_FILENAME, VIDEO_SALIDA
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    CSV_FILENAME, VIDEO_SALIDA = f"pose_log_{stamp}.csv", f"pose_anotado_{stamp}.mp4"

    es_webcam = fuente is None
    cap = cv2.VideoCapture(CAMERA_INDEX if es_webcam else fuente)
    if not cap.isOpened():
        print("ERROR: no se pudo abrir", "la cámara" if es_webcam else f"el video '{fuente}'")
        return None

    fps_src = cap.get(cv2.CAP_PROP_FPS) or 0
    if fps_src <= 1:
        fps_src = 30.0
    inicio = datetime.now()
    registros, hist_muneca = [], deque(maxlen=VELOCITY_WINDOW)
    writer, frame_idx, t_prev, ts_prev = None, 0, 0.0, -1
    t0 = time.monotonic()
    landmarker = crear_landmarker()

    try:
        while cap.isOpened():
            ok, frame = cap.read()
            if not ok:
                break
            h, w = frame.shape[:2]

            t_s = (time.monotonic() - t0) if es_webcam else frame_idx / fps_src
            ts_ms = max(ts_prev + 1, int(t_s * 1000))      # debe ser estrictamente creciente
            fps_inst = 1.0 / max(t_s - t_prev, 1e-3) if frame_idx else fps_src
            t_prev, ts_prev = t_s, ts_ms
            frame_idx += 1

            rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_img, ts_ms)

            if res.pose_landmarks:
                lms = res.pose_landmarks[0]
                V.drawing_utils.draw_landmarks(
                    frame, lms, V.PoseLandmarksConnections.POSE_LANDMARKS,
                    landmark_drawing_spec=V.drawing_styles.get_default_pose_landmarks_style())

                angulos, estados = {}, {}
                for nombre, (p1, p2, p3) in JOINT_ANGLES.items():
                    try:
                        val = round(calcular_angulo(get_coords(lms, p1, w, h),
                                                    get_coords(lms, p2, w, h),
                                                    get_coords(lms, p3, w, h)), 1)
                    except Exception:
                        val = None
                    angulos[nombre] = val
                    estados[nombre] = validar_angulo(nombre, val, enfoque)

                mx, my = get_coords(lms, LANDMARKS.RIGHT_WRIST, w, h)
                hist_muneca.append((t_s, mx, my))
                vel = round(calcular_velocidad(hist_muneca), 1)

                fila = {"timestamp": (inicio + timedelta(seconds=t_s)).isoformat(),
                        "t_s": round(t_s, 3), "enfoque": enfoque,
                        "fps": round(fps_inst, 1), "velocidad_muneca_der_px_s": vel}
                fila.update(angulos)
                fila.update({f"{k}_estado": v for k, v in estados.items()})
                registros.append(fila)

                for i, (nombre, val) in enumerate(angulos.items()):
                    txt = f"{nombre}: {val} deg" if val is not None else f"{nombre}: N/D"
                    cv2.putText(frame, txt, (10, 25 + i * 22), cv2.FONT_HERSHEY_SIMPLEX,
                                0.55, COLORES[estados[nombre]], 1, cv2.LINE_AA)
                cv2.putText(frame, f"Vel. muneca: {vel} px/s", (10, h - 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
            else:
                cv2.putText(frame, "No se detecta persona", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            cv2.putText(frame, f"Enfoque: {enfoque}", (10, h - 12),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

            if guardar_video:
                if writer is None:
                    writer = cv2.VideoWriter(VIDEO_SALIDA, cv2.VideoWriter_fourcc(*"mp4v"), fps_src, (w, h))
                writer.write(frame)

            if mostrar:
                try:
                    cv2.imshow("Pose Tracker (q para salir)", frame)
                    if cv2.waitKey(1) & 0xFF == ord("q"):
                        break
                except cv2.error:
                    mostrar = False
                    print("Sin ventana gráfica disponible; continúo sin vista previa.")
    finally:
        cap.release()
        landmarker.close()
        if writer is not None:
            writer.release()
        try:
            cv2.destroyAllWindows()
        except cv2.error:
            pass

    if not registros:
        print("No se detectó ninguna persona en la fuente.")
        return None
    df = pd.DataFrame(registros)
    df.to_csv(CSV_FILENAME, index=False)
    print(f"Frames leídos: {frame_idx} | con pose detectada: {len(df)}")
    print(f"CSV: {CSV_FILENAME}" + (f" | Video anotado: {VIDEO_SALIDA}" if guardar_video else ""))
    return df

## 8. Grabar con tu webcam desde el navegador (Google Colab)

En Colab el código corre en un servidor, así que `cv2.VideoCapture(0)` no ve tu cámara. Esta celda abre la cámara **en tu navegador**, graba un clip y lo envía al notebook.

- El navegador te pedirá permiso para la cámara: **Permitir**.
- La grabación queda en `sesion.mp4` a 30 fps constantes, lista para `procesar()`.

In [ ]:
import subprocess, shutil
from base64 import b64decode

def convertir_a_h264(origen, destino, fps=None):
    """Convierte a MP4 H.264 (reproducible en el navegador). Con fps fuerza cuadros por segundo constantes."""
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg no está instalado. En Colab ejecuta: !apt-get -qq install ffmpeg")
    filtros = "scale=trunc(iw/2)*2:trunc(ih/2)*2"      # ancho y alto pares, requisito de H.264
    if fps:
        filtros = f"fps={fps}," + filtros
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", origen, "-vf", filtros,
                    "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", destino], check=True)
    return destino


JS_GRABAR = """
async function grabarVideo(segundos, cuentaAtras, ancho, alto) {
  const div = document.createElement('div');
  const info = document.createElement('div');
  info.style.cssText = 'font:14px sans-serif;margin:6px 0;';
  info.textContent = 'Permite el acceso a la cámara...';
  const video = document.createElement('video');
  video.style.cssText = 'display:block;max-width:100%;border-radius:8px;';
  video.muted = true;
  video.playsInline = true;
  const btnIniciar = document.createElement('button');
  btnIniciar.textContent = 'Iniciar grabación';
  const btnDetener = document.createElement('button');
  btnDetener.textContent = 'Detener';
  btnDetener.disabled = true;
  btnDetener.style.marginLeft = '8px';
  div.append(info, video, btnIniciar, btnDetener);
  document.body.appendChild(div);

  const stream = await navigator.mediaDevices.getUserMedia({
    video: {width: {ideal: ancho}, height: {ideal: alto}, frameRate: {ideal: 30}},
    audio: false
  });
  video.srcObject = stream;
  await video.play();
  info.textContent = 'Colócate de modo que se vea el cuerpo completo y pulsa Iniciar.';
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

  await new Promise(r => { btnIniciar.onclick = r; });
  btnIniciar.disabled = true;
  for (let i = cuentaAtras; i > 0; i--) {
    info.textContent = 'Comienza en ' + i + '...';
    await new Promise(r => setTimeout(r, 1000));
  }

  const mime = ['video/webm;codecs=vp9', 'video/webm;codecs=vp8', 'video/webm']
      .find(m => MediaRecorder.isTypeSupported(m)) || '';
  const rec = new MediaRecorder(stream, mime ? {mimeType: mime} : {});
  const chunks = [];
  rec.ondataavailable = e => { if (e.data && e.data.size) chunks.push(e.data); };
  const terminado = new Promise(r => { rec.onstop = r; });
  rec.start();
  btnDetener.disabled = false;

  const t0 = Date.now();
  const timer = setInterval(() => {
    const restante = Math.max(0, segundos - (Date.now() - t0) / 1000);
    info.textContent = 'Grabando... ' + restante.toFixed(0) + ' s';
  }, 250);
  const auto = setTimeout(() => { if (rec.state !== 'inactive') rec.stop(); }, segundos * 1000);
  btnDetener.onclick = () => { if (rec.state !== 'inactive') rec.stop(); };

  await terminado;
  clearInterval(timer);
  clearTimeout(auto);
  stream.getTracks().forEach(t => t.stop());
  info.textContent = 'Enviando video al notebook...';

  const blob = new Blob(chunks, {type: 'video/webm'});
  const b64 = await new Promise(r => {
    const fr = new FileReader();
    fr.onloadend = () => r(fr.result.split(',')[1]);
    fr.readAsDataURL(blob);
  });
  div.remove();
  return b64;
}
"""


def grabar_desde_navegador(segundos=15, cuenta_atras=3, ancho=640, alto=480, salida="sesion.mp4", fps=30):
    """Graba con la webcam del navegador (Colab) y devuelve la ruta de un MP4 listo para procesar()."""
    try:
        from google.colab import output
        from IPython.display import display, Javascript
    except ImportError:
        print("Esta función solo funciona en Google Colab. En local usa procesar(None) para la webcam.")
        return None

    display(Javascript(JS_GRABAR))
    script = f"grabarVideo({segundos}, {cuenta_atras}, {ancho}, {alto})"
    try:
        try:
            b64 = output.eval_js(script, timeout_sec=900)
        except TypeError:                       # versiones de Colab sin timeout_sec
            b64 = output.eval_js(script)
    except Exception as e:
        print("No se pudo grabar:", e)
        print("Revisa que hayas permitido la cámara en el navegador (icono de candado junto a la URL).")
        return None

    if not b64:
        print("No se recibió video.")
        return None
    with open("grabacion.webm", "wb") as f:
        f.write(b64decode(b64))
    convertir_a_h264("grabacion.webm", salida, fps=fps)

    cap = cv2.VideoCapture(salida)
    n, wv, hv = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), int(cap.get(3)), int(cap.get(4))
    cap.release()
    print(f"Video guardado: {salida} | {n} frames, {wv}x{hv} px, ~{n / fps:.1f} s")
    return salida

## 9. Ejecutar (elige una opción)

Quita el `#` de las líneas de la opción que uses. Pasa siempre la ruta a `procesar(...)`.

In [ ]:
# OPCIÓN A — video ya subido a Colab (panel de archivos a la izquierda)
# df_captura = procesar("sesion.mp4")

In [ ]:
# OPCIÓN B — grabar con tu webcam desde el navegador
# 1) ejecuta la celda  2) permite la cámara  3) pulsa "Iniciar grabación"
ruta = grabar_desde_navegador(segundos=30)
df_captura = procesar(ruta)

## 10. Ver el resultado y descargar el CSV (Colab)

In [ ]:
def mostrar_resultados():
    """Reproduce el video anotado en el notebook y descarga el CSV para el dashboard."""
    from IPython.display import Video, display
    reproducible = convertir_a_h264(VIDEO_SALIDA, "anotado_h264.mp4")   # mp4v no se reproduce en navegadores
    display(Video(reproducible, embed=True, width=640))
    try:
        from google.colab import files
        files.download(CSV_FILENAME)
    except ImportError:
        print("CSV guardado en:", CSV_FILENAME)

mostrar_resultados()

## 11. Validación y gráficos

In [ ]:
# Para una captura previa: df_captura = pd.read_csv("pose_log_YYYYMMDD_HHMMSS.csv")

if "df_captura" in globals() and df_captura is not None:
    cols = [c for c in JOINT_ANGLES if c in df_captura.columns]

    print("=== Resumen de validación ===")
    for col in cols:
        pct = df_captura[f"{col}_estado"].value_counts(normalize=True) * 100
        print(f"{col}: " + " | ".join(f"{k} {v:.0f}%" for k, v in pct.items()))

    fig, axes = plt.subplots(len(cols), 1, figsize=(10, 1.8 * len(cols)), sharex=True)
    for ax, col in zip(axes, cols):
        ax.plot(df_captura["t_s"], df_captura[col], linewidth=1)
        lo, hi = RANGOS_REFERENCIA[ENFOQUE][col]
        ax.axhspan(lo, hi, color="green", alpha=0.12)
        ax.set_ylabel(col, fontsize=8)
    axes[-1].set_xlabel("Tiempo (s)")
    plt.suptitle(f"Evolución de ángulos — enfoque: {ENFOQUE}")
    plt.tight_layout()
    plt.show()
else:
    print("Aún no hay datos. Ejecuta procesar() primero.")

## 12. Siguiente paso

Sube el CSV al **dashboard** para cambiar de enfoque sin recapturar y revisar el semáforo por articulación.